# DANFront - First Military Survey of the Banat of Temesvár (1769-1772) tiles

The Temesvár DVD has no georeferencing. Each sheet is placed from its position in the sheet grid (read from the overview
sheet) with an affine transform fitted to 15 towns marked on the sheets (`temes_fit.json`). Cross-checked accuracy:
about 750 m (median), up to ~1.4 km. Tiles go to the R2 bucket under `danfront/tiles-1kf-temes/`.

**Before you start**: the three R2 secrets with *Notebook access* on, and an L4 runtime.
**How to use**: run the cells from top to bottom. After a disconnect run cells 1 to 4 again, then cell 6.

## 1. Setup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
WORK  = '/content/drive/MyDrive/danfront_1kf_temes'   # on Drive: only the list of finished sheets
LOCAL = '/content/work'                          # on the Colab disk: downloaded sheets and tiles (temporary)
os.makedirs(WORK, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)

!pip -q install rasterio pyproj
!command -v 7z >/dev/null || apt-get -qq install -y p7zip-full >/dev/null
!command -v rclone >/dev/null || (curl -s https://rclone.org/install.sh | bash >/dev/null 2>&1)
!7z | sed -n 2p; rclone version | head -1; df -h /content | tail -1
!pip -q install shapely

## 2. Load the scripts
Nothing to change here.

In [ ]:
%%writefile /content/work/tiles_2kf_batch.py
"""
tiles_2kf_batch.py - cut XYZ tiles from many Second Military Survey (2KF) GeoTIFFs at once,
all sheets merged into one tile folder. Used by colab_2kf.ipynb, also runnable by hand.

Input: GeoTIFFs exported with qgis_batch_export.py (Cassini-Soldner, origin St. Stephen's
Cathedral Vienna). Each sheet is clipped automatically to its map frame:
the sheets are 9600 x 6400 Vienna Klafter (18206.24 x 12137.50 m) cells of a regular grid in
that Cassini system, with Vienna in the middle of a cell, i.e. cell (i, j) covers
  X in [(i-0.5)*W, (i+0.5)*W],  Y in [(j-0.5)*H, (j+0.5)*H].
(Checked on 32-50Cas: the nominal cell matches the drawn frame within ~25 m.)
A sheet whose image does not cover most of one cell is tiled unclipped and reported as
"irregular"; files without a CRS are skipped.

Usage:
  python3 tiles_2kf_batch.py <tiles_out_dir> <sheet1.tif> [<sheet2.tif> ...] [--zoom 6,14] [--workers 2]
                             [--bbox lonmin,latmin,lonmax,latmax]

Prints one JSON line per sheet (status, cell, tile bounds) and writes the list of tile files
it created or changed to <tiles_out_dir>/../changed_tiles.txt (relative paths, for
`rclone copy --files-from`). Tiles shared by neighbouring sheets are composited, with a
file lock so parallel workers don't clash.
"""
import argparse, fcntl, json, math, os, sys
from multiprocessing import Pool
import numpy as np
import rasterio
from rasterio.transform import Affine
from rasterio.warp import reproject, transform as warp_transform, Resampling
from rasterio.windows import from_bounds
from PIL import Image

R = 20037508.342789244
KLAFTER = 1.89648384
CELL_W, CELL_H = 9600 * KLAFTER, 6400 * KLAFTER
CASSINI_VIENNA = "+proj=cass +lat_0=48.208772 +lon_0=16.374242 +a=6376130 +rf=310 +units=m +no_defs"


def best_cell(b):
    """Grid cell with the largest overlap with image bounds b; returns (i, j, cell_bounds, covered_fraction_of_cell)."""
    best = None
    ci, cj = round((b.left + b.right) / 2 / CELL_W), round((b.bottom + b.top) / 2 / CELL_H)
    for i in (ci - 1, ci, ci + 1):
        for j in (cj - 1, cj, cj + 1):
            x0, x1, y0, y1 = (i - .5) * CELL_W, (i + .5) * CELL_W, (j - .5) * CELL_H, (j + .5) * CELL_H
            ox = max(0, min(b.right, x1) - max(b.left, x0)); oy = max(0, min(b.top, y1) - max(b.bottom, y0))
            frac = ox * oy / (CELL_W * CELL_H)
            if best is None or frac > best[3]:
                best = (i, j, (x0, y0, x1, y1), frac)
    return best


def unpremultiply(dst):
    """Mosaic pixels are premultiplied (edge pixels were averaged with transparent ones):
    turn them back into straight colour + alpha, so sheet edges don't come out dark."""
    a = dst[3].astype(np.uint16)
    out = dst.copy()
    part = (a > 0) & (a < 255)
    if part.any():
        for c in range(3):
            out[c][part] = np.minimum(255, (dst[c][part].astype(np.uint16) * 255 + a[part] // 2) // a[part])
    return out


def composite_save(path, dst, lock_path):
    """Merge a tile into the tile already on disk. Neighbouring sheets each cover part of an
    edge pixel, so their coverages are ADDED (new sheet first, old fills what is left) instead
    of the usual 'over' blend, which would leave a half-transparent line along every sheet edge."""
    new = unpremultiply(dst).transpose(1, 2, 0)
    with open(lock_path, "w") as lf:
        fcntl.flock(lf, fcntl.LOCK_EX)
        if os.path.exists(path):
            try:
                old = np.asarray(Image.open(path).convert("RGBA"))
                an = new[..., 3:].astype(np.float32)
                ao = np.minimum(old[..., 3:].astype(np.float32), 255 - an)
                tot = an + ao
                rgb = (new[..., :3] * an + old[..., :3] * ao) / np.maximum(tot, 1)
                new = np.concatenate([np.rint(rgb), tot], axis=2).astype(np.uint8)
            except OSError:
                pass                      # damaged tile left by an interrupted run: replace it
        os.makedirs(os.path.dirname(path), exist_ok=True)
        tmp = path + ".tmp"               # write to a temp file and rename, so an interruption
        Image.fromarray(np.ascontiguousarray(new), "RGBA").save(tmp, format="PNG", compress_level=6)   # can never leave a half-written tile behind
        os.replace(tmp, path)


def process(job):
    tif, out_dir, zmin, zmax, bbox = job
    sheet = os.path.splitext(os.path.basename(tif))[0]
    res = dict(sheet=sheet, status="", tiles=[], bounds={})
    try:
        with rasterio.open(tif) as ds:
            if ds.crs is None:
                res["status"] = "skipped: no CRS (RAW sheet)"; return res
            cas = rasterio.crs.CRS.from_proj4(CASSINI_VIENNA)
            same = abs(ds.crs.to_dict().get("lat_0", 0) - 48.208772) < 1e-6 and ds.crs.to_dict().get("proj") == "cass"
            if not same:
                res["status"] = f"skipped: unexpected CRS {ds.crs.to_proj4()[:60]}"; return res
            i, j, (x0, y0, x1, y1), frac = best_cell(ds.bounds)
            res["cell"] = [i, j]; res["cell_coverage"] = round(frac, 3)
            if frac >= 0.9:
                win = from_bounds(x0, y0, x1, y1, ds.transform).round_offsets().round_lengths()
                win = win.intersection(rasterio.windows.Window(0, 0, ds.width, ds.height))
                res["status"] = "ok"
            else:
                win = rasterio.windows.Window(0, 0, ds.width, ds.height)
                res["status"] = "ok, irregular (not clipped)"
            data = ds.read([1, 2, 3], window=win)
            src_tf = ds.window_transform(win); src_crs = ds.crs
        rgba = np.zeros((4,) + data.shape[1:], np.uint8)
        rgba[:3] = data; rgba[3] = np.where(data.any(axis=0), 255, 0); del data
        H, W = rgba.shape[1:]
        n = 20
        edge = ([(W*k/n, 0) for k in range(n+1)] + [(W, H*k/n) for k in range(n+1)] + [(W*k/n, H) for k in range(n+1)] + [(0, H*k/n) for k in range(n+1)])
        xs, ys = zip(*[src_tf * p for p in edge])
        mx, my = warp_transform(src_crs, "EPSG:3857", xs, ys)
        minX, maxX, minY, maxY = min(mx), max(mx), min(my), max(my)
        if bbox:
            lon, lat = warp_transform("EPSG:3857", "EPSG:4326", [minX, maxX], [minY, maxY])
            if lon[1] < bbox[0] or lon[0] > bbox[2] or lat[1] < bbox[1] or lat[0] > bbox[3]:
                res["status"] = "skipped: outside bbox"; return res
        _xs, _ys = zip(*[src_tf * p for p in ((0, 0), (W, 0), (W, H), (0, H))])
        _cx, _cy = warp_transform(src_crs, "EPSG:4326", _xs, _ys)
        res["outline"] = [[round(a, 5), round(b, 5)] for a, b in zip(_cx, _cy)]   # NW, NE, SE, SW (lon, lat)
        lock_path = os.path.join(os.path.dirname(os.path.abspath(out_dir)), ".tiles.lock")
        # One warp per zoom level into a mosaic aligned to the tile grid, then slice it into tiles.
        # The deepest zoom is warped from the sheet; each shallower zoom is averaged from the one above it.
        prev = prev_tf = None
        for z in range(zmax, zmin - 1, -1):
            size = 2 * R / 2 ** z
            xa, xb = int((minX + R) // size), int((maxX + R) // size)
            ya, yb = int((R - maxY) // size), int((R - minY) // size)
            res["bounds"][z] = dict(xMin=xa, xMax=xb, yMin=ya, yMax=yb)
            mos = np.zeros((4, (yb - ya + 1) * 256, (xb - xa + 1) * 256), np.uint8)
            mtf = Affine(size / 256, 0, -R + xa * size, 0, -size / 256, R - ya * size)
            if prev is None:
                reproject(rgba, mos, src_transform=src_tf, src_crs=src_crs, dst_transform=mtf, dst_crs="EPSG:3857",
                          resampling=Resampling.bilinear)
            else:
                reproject(prev, mos, src_transform=prev_tf, src_crs="EPSG:3857", dst_transform=mtf, dst_crs="EPSG:3857",
                          resampling=Resampling.average)
            prev, prev_tf = mos, mtf
            for x in range(xa, xb + 1):
                for y in range(ya, yb + 1):
                    dst = mos[:, (y - ya) * 256:(y - ya + 1) * 256, (x - xa) * 256:(x - xa + 1) * 256]
                    if dst[3].max() == 0:
                        continue
                    rel = f"{z}/{x}/{y}.png"
                    composite_save(os.path.join(out_dir, rel), np.ascontiguousarray(dst), lock_path)
                    res["tiles"].append(rel)
    except Exception as e:
        res["status"] = f"ERROR: {type(e).__name__}: {e}"
    return res


def run(tifs, out_dir, zmin=6, zmax=14, workers=2, bbox=None):
    """Process the given GeoTIFFs; returns the list of per-sheet result dicts."""
    os.makedirs(out_dir, exist_ok=True)
    jobs = [(t, out_dir, zmin, zmax, bbox) for t in tifs]
    if workers > 1 and len(jobs) > 1:
        with Pool(workers) as p:
            results = p.map(process, jobs, chunksize=1)
    else:
        results = [process(j) for j in jobs]
    changed = sorted({t for r in results for t in r["tiles"]})
    with open(os.path.join(os.path.dirname(os.path.abspath(out_dir)), "changed_tiles.txt"), "w") as f:
        f.write("\n".join(changed) + ("\n" if changed else ""))
    return results


def merge_bounds(total, b):
    for z, v in b.items():
        z = int(z); t = total.setdefault(z, dict(v))
        t.update(xMin=min(t["xMin"], v["xMin"]), xMax=max(t["xMax"], v["xMax"]), yMin=min(t["yMin"], v["yMin"]), yMax=max(t["yMax"], v["yMax"]))
    return total


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("out_dir"); ap.add_argument("tifs", nargs="+")
    ap.add_argument("--zoom", default="6,14"); ap.add_argument("--workers", type=int, default=2); ap.add_argument("--bbox")
    a = ap.parse_args()
    zmin, zmax = map(int, a.zoom.split(","))
    bbox = tuple(map(float, a.bbox.split(","))) if a.bbox else None
    total = {}
    for r in run(a.tifs, a.out_dir, zmin, zmax, a.workers, bbox):
        merge_bounds(total, r["bounds"])
        print(json.dumps({k: (len(v) if k == "tiles" else v) for k, v in r.items() if k != "bounds"}, ensure_ascii=False))
    print("tile bounds:", json.dumps(total))


In [ ]:
%%writefile /content/work/banat_frame.py
"""Map area of a First Military Survey sheet of the Banat of Temesvár. These sheets have one thick outer frame that
also encloses the title strip (top) and the column with place names (right); the map is the part below the title
line and left of the column line. Sheets differ in shape, so no fixed width/height ratio is assumed."""
import numpy as np, rasterio, warnings
warnings.filterwarnings('ignore')

def _best(prof, lo, hi):
    lo, hi = max(0, int(lo)), min(len(prof), int(hi))
    if hi <= lo: return None, 0.0
    i = lo + int(np.argmax(prof[lo:hi])); return i, float(prof[i])

def find_map_area(path, step=4):
    """-> dict(W, H, x=(x0, x1), y=(y0, y1), outer=(l, r, t, b), score) in scan pixels, or x/y None."""
    with rasterio.open(path) as ds:
        a = ds.read([1, 2, 3], out_shape=(3, ds.height // step, ds.width // step)).astype(np.int16)
        W, H = ds.width, ds.height
    dark = a.max(axis=0) < 110; h, w = dark.shape
    col = dark.mean(axis=0); row = dark.mean(axis=1)
    l, sl = _best(col, 0, 0.15 * w); r, sr = _best(col, 0.85 * w, w)
    t, st = _best(row, 0, 0.15 * h); b, sb = _best(row, 0.85 * h, h)
    out = dict(W=W, H=H, x=None, y=None, outer=None, score=None, fallback=None)
    sci = 0.0
    if None in (l, r, t, b) or min(sl, sr, st, sb) < 0.35: return out
    # profiles between the outer lines only
    inner = dark[t + 3:b - 2, l + 3:r - 2]
    c2 = inner.mean(axis=0); r2 = inner.mean(axis=1)
    ti, sti = _best(r2, 150 // step, 1000 // step)                     # title line: first strong line below the top
    ci = None
    win = c2[(r - l) - 1500 // step:(r - l) - 250 // step]
    if len(win):
        thr = 0.6 * win.max()
        ci = (r - l) - 1500 // step + int(np.argmax(win >= thr)); sci = float(c2[ci])   # leftmost strong line of the name column
    fallback = []
    if ti is None or sti < 0.45: ti = 380 // step; fallback.append('title')          # typical distance outer line -> title line
    if ci is None or sci < 0.3:  ci = (r - l) - 790 // step; fallback.append('column')  # typical width of the name column
    out['fallback'] = fallback
    x0, x1, y0, y1 = l, l + 3 + ci, t + 3 + ti, b
    out.update(x=(x0 * step, x1 * step), y=(y0 * step, y1 * step), outer=(l * step, r * step, t * step, b * step),
               score=round(min(sl, sr, st, sb), 2))
    return out

if __name__ == '__main__':
    import sys
    for p in sys.argv[1:]:
        print(p.split('/')[-1], find_map_area(p))


In [ ]:
%%writefile /content/work/temes_grid_cells.py
"""Sheet name -> (column, row) of the First Military Survey of the Banat of Temesvár (1769-1772, 208 sheets + 5 1/2),
read from the overview sheet Temes-attekinto (numbers written in the top-left corner of each cell, row by row)."""
ROWS = {  # row: (first sheet number, first column, number of sheets)
    0: (1, 2, 2), 1: (3, 1, 3), 2: (6, 1, 5), 3: (11, 1, 7), 4: (18, 1, 8), 5: (26, 1, 10), 6: (36, 0, 12), 7: (48, 0, 13),
    8: (61, 0, 13), 9: (74, 1, 12), 10: (86, 1, 12), 11: (98, 0, 13), 12: (111, 0, 13), 13: (124, 0, 13), 14: (137, 0, 13),
    15: (150, 0, 13), 16: (163, 1, 12), 17: (175, 1, 11), 18: (186, 5, 7), 19: (193, 6, 5), 20: (198, 6, 5), 21: (203, 7, 4),
    22: (207, 7, 2)}
CELL = {}
for r, (n0, c0, k) in ROWS.items():
    for i in range(k):
        CELL[f'{n0 + i:03d}'] = (c0 + i, r)
CELL['005-1_2'] = (4, 1)          # the extra half sheet "5 1/2" east of sheet 5
assert len(CELL) == 209 and CELL['208'] == (8, 22) and CELL['060'] == (12, 7) and CELL['185'] == (11, 17)


In [ ]:
%%writefile /content/work/temes_fit.json
{
 "local": "+proj=aeqd +lat_0=45.5 +lon_0=21.3 +datum=WGS84 +units=m",
 "model": "affine on (column + fx, row + fy)",
 "coef": [
  [
   15011.22671124707,
   4729.316744511343
  ],
  [
   3202.5775740662775,
   -10012.941527214874
  ],
  [
   -127970.02750646949,
   74735.21015864235
  ]
 ],
 "accuracy_m": {
  "loo_median": 755,
  "loo_max": 1395
 },
 "control_points": [
  {
   "name": "Lugos",
   "sheet": "082",
   "px": 4646.0,
   "py": 4929.25,
   "lat": 45.6886,
   "lon": 21.9031,
   "fx": [
    112,
    7480
   ],
   "fy": [
    556,
    5516
   ]
  },
  {
   "name": "Lippa",
   "sheet": "034",
   "px": 2775.25,
   "py": 2125.75,
   "lat": 46.0886,
   "lon": 21.6936,
   "fx": [
    144,
    7448
   ],
   "fy": [
    540,
    5532
   ]
  },
  {
   "name": "Csanad",
   "sheet": "009",
   "px": 2125.0,
   "py": 2898.5,
   "lat": 46.1333,
   "lon": 20.5833,
   "fx": [
    112,
    7448
   ],
   "fy": [
    568,
    5572
   ]
  },
  {
   "name": "Zsombolya",
   "sheet": "040",
   "px": 1593.75,
   "py": 1992.1875,
   "lat": 45.7914,
   "lon": 20.7172,
   "fx": [
    188,
    7500
   ],
   "fy": [
    576,
    5536
   ]
  },
  {
   "name": "Nagybecskerek",
   "sheet": "074",
   "px": 6938.375,
   "py": 3527.125,
   "lat": 45.38,
   "lon": 20.39,
   "fx": [
    120,
    8596
   ],
   "fy": [
    576,
    5608
   ]
  },
  {
   "name": "Pancsova",
   "sheet": "151",
   "px": 5842.4375,
   "py": 1855.6875,
   "lat": 44.8708,
   "lon": 20.6403,
   "fx": [
    148,
    7600
   ],
   "fy": [
    608,
    5652
   ]
  },
  {
   "name": "Feherttemplom",
   "sheet": "167",
   "px": 4185.25,
   "py": 4828.875,
   "lat": 44.8925,
   "lon": 21.4178,
   "fx": [
    140,
    7596
   ],
   "fy": [
    592,
    5608
   ]
  },
  {
   "name": "Versec",
   "sheet": "142",
   "px": 3623.25,
   "py": 1932.125,
   "lat": 45.1167,
   "lon": 21.3036,
   "fx": [
    128,
    7532
   ],
   "fy": [
    576,
    5504
   ]
  },
  {
   "name": "Kevevara",
   "sheet": "177",
   "px": 929.6875,
   "py": 2216.0,
   "lat": 44.7475,
   "lon": 20.9761,
   "fx": [
    140,
    7796
   ],
   "fy": [
    612,
    5676
   ]
  },
  {
   "name": "Alibunar",
   "sheet": "127",
   "px": 6131.1875,
   "py": 5110.75,
   "lat": 45.0806,
   "lon": 20.9658,
   "fx": [
    88,
    7536
   ],
   "fy": [
    568,
    5496
   ]
  },
  {
   "name": "Karansebes",
   "sheet": "134",
   "px": 3745.75,
   "py": 2104.3125,
   "lat": 45.4144,
   "lon": 22.2208,
   "fx": [
    132,
    7596
   ],
   "fy": [
    548,
    5440
   ]
  },
  {
   "name": "Omoldova",
   "sheet": "193",
   "px": 1673.4375,
   "py": 1089.0625,
   "lat": 44.7231,
   "lon": 21.6292,
   "fx": [
    256,
    6828
   ],
   "fy": [
    1068,
    5588
   ]
  },
  {
   "name": "Ecska",
   "sheet": "086",
   "px": 7576.9375,
   "py": 2295.0625,
   "lat": 45.32,
   "lon": 20.44,
   "fx": [
    172,
    8780
   ],
   "fy": [
    560,
    5584
   ]
  },
  {
   "name": "Novi Knezevac",
   "sheet": "006",
   "px": 6520.125,
   "py": 1373.3125,
   "lat": 46.0444,
   "lon": 20.1,
   "fx": [
    132,
    7528
   ],
   "fy": [
    588,
    5564
   ]
  },
  {
   "name": "Belgrade fortress opposite (Pancsova bank)",
   "sheet": "150",
   "px": 5856.75,
   "py": 1878.375,
   "lat": 44.8231,
   "lon": 20.4504,
   "fx": [
    144,
    7572
   ],
   "fy": [
    588,
    5612
   ]
  }
 ]
}

In [ ]:
%%writefile /content/work/tiles_temes_batch.py
"""
tiles_temes_batch.py - tiles for the First Military Survey of the Banat of Temesvár (1769-1772, Arcanum DVD without
georeferencing). Same interface as tiles_2kf_batch.py.

How a sheet is placed:
  1. banat_frame.py finds the map area (below the title line, left of the place-name column).
  2. The sheets form a grid (temes_grid_cells.py: sheet name -> column, row), read from the overview sheet.
  3. temes_fit.json: affine transform from (column + fraction across the map, row + fraction down) to the ground,
     fitted to 15 towns marked on the sheets. Cross-checked accuracy: about 750 m (median), up to ~1.4 km.
Call configure(fit_json) once before run().
"""
import json, os, re
from multiprocessing import Pool
import numpy as np
import rasterio
from rasterio.control import GroundControlPoint
from rasterio.transform import Affine
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window
from pyproj import Transformer
from tiles_2kf_batch import composite_save, merge_bounds, R
from temes_grid_cells import CELL
from banat_frame import find_map_area

CFG = {}
INSET = 14          # half the thickness of the drawn frame lines, in scan pixels

def configure(fit_json):
    d = json.load(open(fit_json)); CFG.clear(); CFG.update(d); CFG['coef'] = np.array(d['coef'])
    CFG['to_merc'] = Transformer.from_crs(d['local'], 'EPSG:3857', always_xy=True)
    return len(d['control_points'])

def grid_to_local(UV):
    UV = np.atleast_2d(np.asarray(UV, float)); return np.c_[UV, np.ones(len(UV))] @ CFG['coef']

def process(job):
    tif, out_dir, zmin, zmax, bbox = job
    sheet = os.path.splitext(os.path.basename(tif))[0]
    res = dict(sheet=sheet, status="", tiles=[], bounds={})
    try:
        if sheet not in CELL:
            res["status"] = "skipped: not a sheet of the grid"; return res
        fr = find_map_area(tif)
        if not (fr['x'] and fr['y']):
            res["status"] = f"skipped: map frame not found ({fr['W']}x{fr['H']})"; return res
        (fx0, fx1), (fy0, fy1) = fr['x'], fr['y']
        x0, y0, x1, y1 = fx0 + INSET, fy0 + INSET, fx1 - INSET, fy1 - INSET
        W, H = x1 - x0, y1 - y0
        c, r = CELL[sheet]
        k = 7; A = []; UU = []
        for i in range(k):
            for j in range(k):
                px, py = W * i / (k - 1), H * j / (k - 1)                       # pixel in the tiled window
                fx = (x0 + px - fx0) / (fx1 - fx0); fy = (y0 + py - fy0) / (fy1 - fy0)   # position in frame units (beyond 0..1 on a glued piece)
                A.append((px, py, 1)); UU.append((c + fx, r + fy))
        L = grid_to_local(UU); B = np.array(CFG['to_merc'].transform(L[:, 0], L[:, 1])).T
        A = np.array(A)
        # the correction bends a little inside one sheet: warp with a 3rd-order polynomial through the 7x7 mesh
        # (GDAL chooses it from the control points), so neighbouring sheets meet along their common edge
        gcps = [GroundControlPoint(row=float(py), col=float(px), x=float(X), y=float(Y)) for (px, py, _), (X, Y) in zip(A, B)]
        x_, y_ = A[:, 0] / W, A[:, 1] / H
        D = np.stack([np.ones_like(x_), x_, y_, x_*x_, x_*y_, y_*y_, x_**3, x_*x_*y_, x_*y_*y_, y_**3], 1)
        co, *_ = np.linalg.lstsq(D, B, rcond=None)
        res["warp_residual_m"] = round(float(np.hypot(*(D @ co - B).T).max()) * 0.69, 1)        # 0.69 ~ cos(46 deg): Mercator -> metres
        cs = [tuple(B[i]) for i in (0, (k - 1) * k, k * k - 1, k - 1)]                              # NW, NE, SE, SW
        edge = [tuple(B[i * k + j]) for i in range(k) for j in range(k) if i in (0, k - 1) or j in (0, k - 1)]
        to_wgs = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
        res["outline"] = [[round(v, 5) for v in to_wgs.transform(*p)] for p in cs]
        minX, maxX = min(p[0] for p in edge), max(p[0] for p in edge); minY, maxY = min(p[1] for p in edge), max(p[1] for p in edge)
        with rasterio.open(tif) as ds:
            data = ds.read([1, 2, 3], window=Window(x0, y0, W, H))
        rgba = np.zeros((4,) + data.shape[1:], np.uint8)
        rgba[:3] = data; rgba[3] = 255
        del data
        res["status"] = "ok" + (f", typical {' and '.join(fr['fallback'])} position used" if fr.get('fallback') else "")
        lock_path = os.path.join(os.path.dirname(os.path.abspath(out_dir)), ".tiles.lock")
        prev = prev_tf = None
        for z in range(zmax, zmin - 1, -1):
            size = 2 * R / 2 ** z
            xa, xb = int((minX + R) // size), int((maxX + R) // size)
            ya, yb = int((R - maxY) // size), int((R - minY) // size)
            res["bounds"][z] = dict(xMin=xa, xMax=xb, yMin=ya, yMax=yb)
            mos = np.zeros((4, (yb - ya + 1) * 256, (xb - xa + 1) * 256), np.uint8)
            mtf = Affine(size / 256, 0, -R + xa * size, 0, -size / 256, R - ya * size)
            if prev is None:
                reproject(rgba, mos, gcps=gcps, src_crs="EPSG:3857", dst_transform=mtf, dst_crs="EPSG:3857", resampling=Resampling.bilinear)
            else:
                reproject(prev, mos, src_transform=prev_tf, src_crs="EPSG:3857", dst_transform=mtf, dst_crs="EPSG:3857", resampling=Resampling.average)
            prev, prev_tf = mos, mtf
            for x in range(xa, xb + 1):
                for y in range(ya, yb + 1):
                    dst = mos[:, (y - ya) * 256:(y - ya + 1) * 256, (x - xa) * 256:(x - xa + 1) * 256]
                    if dst[3].max() == 0: continue
                    rel = f"{z}/{x}/{y}.png"
                    composite_save(os.path.join(out_dir, rel), np.ascontiguousarray(dst), lock_path)
                    res["tiles"].append(rel)
    except Exception as e:
        res["status"] = f"ERROR: {type(e).__name__}: {e}"
    return res

def run(tifs, out_dir, zmin=6, zmax=14, workers=2, bbox=None):
    assert CFG, "call configure(fit_json) first"
    os.makedirs(out_dir, exist_ok=True)
    jobs = [(t, out_dir, zmin, zmax, bbox) for t in tifs]
    if workers > 1 and len(jobs) > 1:
        with Pool(workers) as p: results = p.map(process, jobs, chunksize=1)
    else:
        results = [process(j) for j in jobs]
    changed = sorted({t for r in results for t in r["tiles"]})
    with open(os.path.join(os.path.dirname(os.path.abspath(out_dir)), "changed_tiles.txt"), "w") as f:
        f.write("\n".join(changed) + ("\n" if changed else ""))
    return results


In [ ]:
%%writefile /content/work/danfront_funcs.py
# ---- helper functions (pure Python; no Colab-specific calls) ----
import csv, json, math, os, shutil, subprocess, sys, time
import requests

def free_gb(path):
    return shutil.disk_usage(path).free / 1e9

def wait_for_disk(path, min_free=12, resume_at=20):
    """Pause while the local disk is nearly full (Drive needs time to sync its write cache)."""
    if free_gb(path) >= min_free:
        return
    print(f"  local disk low ({free_gb(path):.0f} GB free) - waiting for Drive to catch up ...")
    while free_gb(path) < resume_at:
        time.sleep(30)
    print("  continuing")

def download_parts(url, dest_prefix, part_size=4 * 1024**3, chunk=8 * 1024**2, disk_path="/content"):
    """Download url as raw volumes dest_prefix.001, .002, ... (7-Zip opens these directly).
    Finished parts are skipped, so the cell can be re-run with a fresh link after an interruption."""
    s = requests.Session()
    r = s.get(url, headers={"Range": "bytes=0-0"}, stream=True, timeout=60)
    cr = r.headers.get("Content-Range"); status = r.status_code; ctype = r.headers.get("Content-Type", ""); r.close()
    if status in (401, 403) or "text/html" in ctype:
        raise RuntimeError(f"The link was refused (HTTP {status}, {ctype}). It has probably expired: copy a fresh download link and run this cell again.")
    if status != 206 or not cr:
        raise RuntimeError(f"The server did not accept a ranged request (HTTP {status}). Send this message to Claude.")
    total = int(cr.split("/")[-1])
    size_file = dest_prefix + ".size"
    if os.path.exists(size_file):
        old = int(open(size_file).read().strip())
        if old != total:
            raise RuntimeError(f"This link points to a file of {total} bytes, but the parts already downloaded belong to a file of {old} bytes. Wrong link?")
    else:
        open(size_file, "w").write(str(total))
    nparts = math.ceil(total / part_size)
    print(f"archive size {total/1e9:.1f} GB -> {nparts} parts")
    t0 = time.time(); done_bytes = 0; new_bytes = 0
    for k in range(nparts):
        start = k * part_size; end = min(total, start + part_size) - 1; want = end - start + 1
        part = f"{dest_prefix}.{k+1:03d}"
        if os.path.exists(part) and os.path.getsize(part) == want:
            done_bytes += want; continue
        tmp = part + ".tmp"
        for attempt in range(5):
            try:
                r = s.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=120)
                if r.status_code != 206:
                    raise RuntimeError(f"HTTP {r.status_code}")
                got = 0
                with open(tmp, "wb") as f:
                    for c in r.iter_content(chunk):
                        f.write(c); got += len(c)
                        wait_for_disk(disk_path)
                if got != want:
                    raise IOError(f"got {got} of {want} bytes")
                os.replace(tmp, part); break
            except Exception as e:
                print(f"  part {k+1}: {e} - retrying ({attempt+1}/5)"); time.sleep(20)
        else:
            raise RuntimeError("Download keeps failing. The link has probably expired: copy a fresh download link and run this cell again (finished parts are kept).")
        done_bytes += want; new_bytes += want
        rate = new_bytes / max(1, time.time() - t0) / 1e6
        print(f"part {k+1}/{nparts} done  ({done_bytes/1e9:.1f} of {total/1e9:.1f} GB, {rate:.0f} MB/s)")
    have = sum(os.path.getsize(f"{dest_prefix}.{k+1:03d}") for k in range(nparts))
    assert have == total, f"size mismatch: {have} != {total}"
    print("download complete and verified:", f"{total/1e9:.1f} GB")

def list_archive(first_volume, exe="7z"):
    """Return (properties dict, [(path, size)] of GeoTIFFs with coordinates) from the archive listing."""
    out = subprocess.run([exe, "l", "-slt", first_volume], capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError("7z could not read the archive:\n" + out.stdout[-2000:] + out.stderr[-2000:])
    head, _, body = out.stdout.partition("----------")
    props = dict(l.split(" = ", 1) for l in head.splitlines() if " = " in l)
    sheets = []
    for block in body.split("\n\n"):
        e = dict(l.split(" = ", 1) for l in block.splitlines() if " = " in l)
        p = e.get("Path", "")
        if not p.lower().endswith((".tif", ".tiff")) or e.get("Folder") == "+" or e.get("Attributes", "").startswith("D"):
            continue
        if "/raw/" in "/" + p.replace("\\", "/"):
            continue                      # RAW sheets (no coordinates) are not for this notebook
        sheets.append((p, int(e.get("Size", 0))))
    return props, sorted(sheets)

def load_done(done_csv):
    if not os.path.exists(done_csv):
        return {}
    with open(done_csv, newline="", encoding="utf-8") as f:
        return {r["path"]: r for r in csv.DictReader(f)}

DONE_FIELDS = ["path", "sheet", "status", "cell_i", "cell_j", "cell_coverage", "tiles"]

def run_batches(cfg, max_sheets=None, batch_size=12, bbox=None, workers=2):
    """Extract -> tile -> upload -> record, batch by batch. Safe to interrupt and re-run."""
    import importlib
    tb = importlib.import_module(cfg.get("tiler", "tiles_2kf_batch"))   # tiles_1kf_batch for the First Survey
    done = load_done(cfg["done_csv"])
    todo = [p for p, _ in cfg["sheets"] if p not in done]
    if max_sheets:
        todo = todo[:max_sheets]
    print(f"{len(done)} sheets already done, {len(todo)} to do in this run")
    if (done or cfg.get("restore_always")) and not os.path.exists(cfg["restored_marker"]):
        print("new session: fetching the tiles made so far back from R2 (needed to merge sheet edges) ...")
        rc = subprocess.run(["rclone", "copy", cfg["remote"], cfg["tiles"], "--transfers", "64", "--checkers", "64"]).returncode
        if rc != 0:
            raise RuntimeError("could not restore tiles from R2")
    os.makedirs(cfg["tiles"], exist_ok=True); open(cfg["restored_marker"], "w").close()
    bounds = json.load(open(cfg["bounds_json"])) if os.path.exists(cfg["bounds_json"]) else {}
    bounds = {int(z): b for z, b in bounds.items()}
    t0 = time.time(); n_done = 0
    for b0 in range(0, len(todo), batch_size):
        batch = todo[b0:b0 + batch_size]
        wait_for_disk(cfg["disk_path"])
        shutil.rmtree(cfg["extract"], ignore_errors=True); os.makedirs(cfg["extract"])
        if cfg.get("fetch"):              # sheets come as single files (e.g. from a SharePoint folder)
            cfg["fetch"](batch, cfg["extract"])
        else:                             # sheets come out of the 7z archive
            lst = os.path.join(cfg["local"], "batch.txt")
            open(lst, "w", encoding="utf-8").write("\n".join(batch) + "\n")
            ex = subprocess.run([cfg.get("exe", "7z"), "x", cfg["first_volume"], f"-o{cfg['extract']}", f"-i@{lst}", "-y", "-bso0", "-bsp0"],
                                capture_output=True, text=True)
            if ex.returncode not in (0, 1):
                raise RuntimeError("7z extraction failed:\n" + ex.stdout[-1500:] + ex.stderr[-1500:])
        tifs, missing = [], []
        for p in batch:
            fp = os.path.join(cfg["extract"], p.replace("\\", "/"))
            (tifs if os.path.exists(fp) else missing).append((p, fp))
        results = tb.run([fp for _, fp in tifs], cfg["tiles"], 6, 14, workers=workers, bbox=bbox)
        changed = os.path.join(os.path.dirname(os.path.abspath(cfg["tiles"])), "changed_tiles.txt")
        if os.path.getsize(changed) > 0:
            for attempt in range(3):
                rc = subprocess.run(["rclone", "copy", cfg["tiles"], cfg["remote"], "--files-from", changed, "--no-traverse",
                                     "--transfers", "32"]).returncode
                if rc == 0:
                    break
                print("  upload failed, retrying ..."); time.sleep(15)
            else:
                raise RuntimeError("Upload to R2 failed 3 times. Check the R2 keys, then run this cell again.")
        rows = []
        for (p, _), r in zip(tifs, results):
            tb.merge_bounds(bounds, r["bounds"])
            c = r.get("cell", ["", ""])
            rows.append(dict(path=p, sheet=r["sheet"], status=r["status"], cell_i=c[0], cell_j=c[1],
                             cell_coverage=r.get("cell_coverage", ""), tiles=len(r["tiles"])))
        for p, _ in missing:
            rows.append(dict(path=p, sheet=os.path.basename(p), status="ERROR: not extracted", cell_i="", cell_j="", cell_coverage="", tiles=0))
        op = cfg.get("outlines_json") or os.path.join(os.path.dirname(cfg["done_csv"]), "sheet_outlines.json")
        outlines = json.load(open(op)) if os.path.exists(op) else {}
        outlines.update({r["sheet"]: r["outline"] for r in results if r.get("outline") and r["status"].startswith("ok")})
        json.dump(outlines, open(op, "w"))          # sheet name -> corners (lon, lat), for the map's sheet lookup
        new_file = not os.path.exists(cfg["done_csv"])
        with open(cfg["done_csv"], "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=DONE_FIELDS)
            if new_file:
                w.writeheader()
            w.writerows(rows)
        json.dump(bounds, open(cfg["bounds_json"], "w"))
        shutil.rmtree(cfg["extract"], ignore_errors=True)
        n_done += len(batch)
        el = time.time() - t0; left = (len(todo) - n_done) * el / n_done
        bad = [r["sheet"] + ": " + r["status"] for r in rows if not r["status"].startswith(("ok", "skipped: already", "skipped: outside"))]
        print(f"[{n_done}/{len(todo)}] batch done in total {el/60:.0f} min, about {left/60:.0f} min left" + (f"   notes: {bad}" if bad else ""))
    print("run finished")

def stitch_preview(tiles_dir, z, max_px=1600):
    """Stitch all local tiles of zoom z into one image (for a quick look)."""
    import glob
    from PIL import Image
    tiles = glob.glob(os.path.join(tiles_dir, str(z), "*", "*.png"))
    if not tiles:
        return None
    key = lambda p: (int(p.split(os.sep)[-2]), int(os.path.basename(p)[:-4]))
    xs = sorted({key(p)[0] for p in tiles}); ys = sorted({key(p)[1] for p in tiles})
    W = Image.new("RGBA", (256 * (xs[-1] - xs[0] + 1), 256 * (ys[-1] - ys[0] + 1)), (225, 225, 225, 255))
    for p in tiles:
        x, y = key(p); W.alpha_composite(Image.open(p).convert("RGBA"), ((x - xs[0]) * 256, (y - ys[0]) * 256))
    W = W.convert("RGB")
    if max(W.size) > max_px:
        W.thumbnail((max_px, max_px))
    return W

def summary(cfg):
    done = load_done(cfg["done_csv"])
    from collections import Counter
    c = Counter(r["status"].split(":")[0] for r in done.values())
    print("sheets:", len(done), dict(c))
    for r in done.values():
        if (not r["status"].startswith(("ok", "skipped: already", "skipped: outside"))) or "irregular" in r["status"] or "not clipped" in r["status"]:
            print("  ", r["sheet"], "-", r["status"], "coverage", r["cell_coverage"])
    b = json.load(open(cfg["bounds_json"])) if os.path.exists(cfg["bounds_json"]) else {}
    print("\n--- copy everything below this line to Claude ---")
    print("sheets_ok:", sum(1 for r in done.values() if r["status"].startswith("ok")), "of", len(done))
    print(f"var {cfg.get('bounds_var', 'kf2TileBounds')} = {{")
    for z in sorted(b, key=int):
        v = b[z]; print(f"    {int(z)}: {{ xMin: {v['xMin']}, xMax: {v['xMax']}, yMin: {v['yMin']}, yMax: {v['yMax']} }},")
    print("};")

def repair(cfg):
    """After an interrupted run: fix half-written tiles and put failed sheets back on the to-do list."""
    import glob
    from PIL import Image
    bad = []
    for p in glob.glob(os.path.join(cfg["tiles"], "*", "*", "*.png")):
        try:
            with Image.open(p) as im:
                im.load()
        except Exception:
            bad.append(p)
    print(len(bad), "damaged tiles found")
    for p in bad:
        rel = os.path.relpath(p, cfg["tiles"]).replace(os.sep, "/")
        os.remove(p)
        subprocess.run(["rclone", "copyto", f"{cfg['remote']}/{rel}", p], capture_output=True)   # last good copy from R2, if any
        ok = False
        if os.path.exists(p):
            try:
                with Image.open(p) as im:
                    im.load()
                ok = True
            except Exception:
                os.remove(p)
        print("  ", rel, "- restored from R2" if ok else "- removed (will be rebuilt)")
    for t in glob.glob(os.path.join(cfg["tiles"], "*", "*", "*.tmp")):
        os.remove(t)
    if os.path.exists(cfg["done_csv"]):
        rows = list(csv.DictReader(open(cfg["done_csv"], newline="", encoding="utf-8")))
        keep = [r for r in rows if not r["status"].startswith("ERROR")]
        with open(cfg["done_csv"], "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=DONE_FIELDS); w.writeheader(); w.writerows(keep)
        print(len(rows) - len(keep), "failed sheets put back on the to-do list:", [r["sheet"] for r in rows if r["status"].startswith("ERROR")])


# ---- SharePoint folder access (for packages uploaded as single files, not as one archive) ----
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120 Safari/537.36"

class SharePointFolder:
    """Lists and downloads the files of a folder shared with an 'anyone with the link' sharing link."""
    def __init__(self, share_url):
        self.share_url = share_url
        self.login()

    def login(self):
        from urllib.parse import urlparse, parse_qs
        self.s = requests.Session(); self.s.headers["User-Agent"] = UA
        r = self.s.get(self.share_url, timeout=60)
        u = urlparse(r.url); q = parse_qs(u.query)
        if r.status_code != 200 or "id" not in q:
            raise RuntimeError(f"The sharing link did not open a folder (status {r.status_code}, landed on {r.url[:150]}). "
                               "Check that SHARE is the link of the folder itself.")
        self.root = q["id"][0]                                  # e.g. /personal/<user>/Documents/.../pc60_firstMS
        self.site = f"{u.scheme}://{u.netloc}" + "/".join(self.root.split("/")[:3])

    def _api(self, path, what):
        from urllib.parse import quote
        url = (f"{self.site}/_api/web/GetFolderByServerRelativePath(decodedurl='{quote(path.replace(chr(39), chr(39)*2))}')/{what}"
               "?$top=5000")
        for attempt in range(4):
            r = self.s.get(url, headers={"Accept": "application/json;odata=nometadata"}, timeout=60)
            if r.status_code == 200:
                return r.json()["value"]
            if r.status_code in (401, 403):
                self.login()
            time.sleep(3)
        raise RuntimeError(f"SharePoint did not list {path} (status {r.status_code}): {r.text[:300]}")

    def files(self, sub=""):
        """[(name, size, server_relative_url)] of the files directly in root/sub."""
        p = self.root + ("/" + sub.strip("/") if sub else "")
        return sorted((f["Name"], int(f["Length"]), f["ServerRelativeUrl"]) for f in self._api(p, "Files"))

    def folders(self, sub=""):
        p = self.root + ("/" + sub.strip("/") if sub else "")
        return sorted(f["Name"] for f in self._api(p, "Folders") if f["Name"] != "Forms")

    def download(self, rel_url, size, dest):
        from urllib.parse import quote
        if os.path.exists(dest) and os.path.getsize(dest) == size:
            return
        url = f"{self.site}/_layouts/15/download.aspx?SourceUrl={quote(rel_url)}"
        err = None
        for attempt in range(5):
            try:
                with self.s.get(url, stream=True, timeout=120) as r:
                    if r.status_code in (401, 403):
                        self.login(); raise IOError(f"status {r.status_code}")
                    r.raise_for_status()
                    with open(dest + ".part", "wb") as f:
                        for chunk in r.iter_content(4 * 1024**2):
                            f.write(chunk)
                if os.path.getsize(dest + ".part") != size:
                    raise IOError("size mismatch")
                os.replace(dest + ".part", dest); return
            except Exception as e:
                err = e; time.sleep(5 * (attempt + 1))
        raise RuntimeError(f"could not download {rel_url}: {err}")

    def download_many(self, items, dest_dir, threads=8):
        """items: [(name, size, rel_url)] -> dest_dir/name."""
        from concurrent.futures import ThreadPoolExecutor
        os.makedirs(dest_dir, exist_ok=True)
        with ThreadPoolExecutor(threads) as ex:
            list(ex.map(lambda it: self.download(it[2], it[1], os.path.join(dest_dir, it[0])), items))

    def download_tree(self, sub, dest_dir, threads=16):
        """Download root/sub with all its subfolders (for the small .ini files). Returns the number of files."""
        n = 0
        items = self.files(sub)
        self.download_many(items, dest_dir, threads); n += len(items)
        for d in self.folders(sub):
            n += self.download_tree(f"{sub}/{d}", os.path.join(dest_dir, d), threads)
        return n

    def find_tifs(self, sub="", skip=("raw",), depth=4):
        """All .tif files under root/sub (subfolders included, folders named in skip left out):
        [(relative_path, size, server_relative_url)]."""
        out = [((sub + "/" if sub else "") + n, size, url) for n, size, url in self.files(sub) if n.lower().endswith((".tif", ".tiff"))]
        if depth > 0:
            for d in self.folders(sub):
                if d.lower() not in skip:
                    out += self.find_tifs((sub + "/" if sub else "") + d, skip, depth - 1)
        return out


In [ ]:
import sys, importlib
sys.path.insert(0, LOCAL)
import banat_frame, danfront_funcs as F, tiles_2kf_batch, tiles_temes_batch as T
for m in (banat_frame, tiles_2kf_batch, F, T): importlib.reload(m)
print('control points in the fit:', T.configure(f'{LOCAL}/temes_fit.json'))

## 3. Connect to R2

In [ ]:
os.environ.update({
    'RCLONE_CONFIG_R2_TYPE': 's3',
    'RCLONE_CONFIG_R2_PROVIDER': 'Cloudflare',
    'RCLONE_CONFIG_R2_ACCESS_KEY_ID': userdata.get('R2_ACCESS_KEY_ID').strip(),
    'RCLONE_CONFIG_R2_SECRET_ACCESS_KEY': userdata.get('R2_SECRET_ACCESS_KEY').strip(),
    'RCLONE_CONFIG_R2_ENDPOINT': userdata.get('R2_ENDPOINT').strip(),
    'RCLONE_S3_NO_CHECK_BUCKET': 'true',
})
REMOTE = 'r2:danfront/tiles-1kf-temes'
import subprocess
r = subprocess.run(['rclone', 'lsf', REMOTE + '/'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('R2 could not be reached. Check the three secrets (and that Notebook access is on).\n' + r.stderr[-800:])
print('R2 connection OK. Folders in tiles-1kf-temes:', r.stdout.split())

## 4. Open the SharePoint folder
`SHARE` is the sharing link of the `pc35-firstMS_Trans_Temes` folder.

In [ ]:
SHARE = "PASTE_THE_SHARING_LINK_OF_THE_FOLDER_HERE"
RAW = "1ms_Geotiff_export_Temesvar/raw"

import collections
SP = F.SharePointFolder(SHARE)
print("shared folder:", SP.root.split("/")[-1], "| contains:", SP.folders())
allf = {n: (n, size, url) for n, size, url in SP.files(RAW) if n.lower().endswith(".tif")}
ITEMS = {n: it for n, it in allf.items() if n[:-4] in T.CELL}
print(len(ITEMS), "grid sheets,", round(sum(it[1] for it in ITEMS.values())/1e9, 1), "GB | left out:", sorted(set(allf) - set(ITEMS)))
SHEETS = sorted((n, it[1]) for n, it in ITEMS.items())

def fetch(batch, dest):
    SP.download_many([ITEMS[n] for n in batch], dest)

CFG = dict(sheets=SHEETS, local=LOCAL, extract=f'{LOCAL}/sheets', tiles=f'{LOCAL}/tiles', disk_path='/content',
           done_csv=f'{WORK}/done.csv', bounds_json=f'{WORK}/tile_bounds.json', restored_marker=f'{LOCAL}/.restored',
           remote=REMOTE, tiler='tiles_temes_batch', fetch=fetch, bounds_var='kf1TemesTileBounds')
print(len(F.load_done(CFG['done_csv'])), 'sheets already processed')

## 5. Test with a few sheets
Processes six sheets around Temesvár and shows them stitched together. Send the picture and the messages to Claude before going on.

In [ ]:
test = [(n, s) for n, s in SHEETS if n[:3] in ("053","054","055","066","067","068")]
F.run_batches(dict(CFG, sheets=test), batch_size=6, workers=6)
from IPython.display import display
display(F.stitch_preview(CFG['tiles'], 11, max_px=1100))
for r in F.load_done(CFG['done_csv']).values(): print(r['sheet'], r['status'])

## 6. Process everything
Safe to interrupt and run again.

In [ ]:
F.run_batches(CFG, batch_size=24, workers=os.cpu_count())

## 7. Result
Copy everything this cell prints and send it to Claude.

In [ ]:
display(F.stitch_preview(CFG['tiles'], 8, max_px=1000))
F.summary(CFG)
done = F.load_done(CFG['done_csv'])
print(collections.Counter(r['status'] for r in done.values()))
print('left out:', sorted((r['sheet'], r['status']) for r in done.values() if not r['status'].startswith('ok')))
print('typical frame position used:', sorted(r['sheet'] for r in done.values() if 'typical' in r['status']))

## 8. Repair (only if needed)

In [ ]:
F.repair(CFG)